In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

In [2]:
df_af2 = pd.read_csv("./contact_overlap_af3_af2.csv")
df_boltz = pd.read_csv("./contact_overlap_af3_boltz.csv")
df_chai1 = pd.read_csv("./contact_overlap_af3_chai1.csv")
df_helixfold3 = pd.read_csv("./contact_overlap_af3_helixfold.csv")
df_of3 = pd.read_csv("./contact_overlap_af3_of3.csv")
df_protenix = pd.read_csv("./contact_overlap_af3_protenix.csv")

df_designclass = pd.read_csv("/Users/elliotttaguchidickinson/Desktop/DeNovoProteinHal/thesis/results/AdaptyvBio/contest_2_round_2_metrics_summary.csv")

In [5]:
df_of3.head()

,Complex,af3_contacts,of3_contacts,overlap_contacts
0,724928537.dbz6___a,69,174,0
1,724928537.dbz7___a,42,121,5
2,724928537.dbz8___a,44,183,0
3,ali.rasouli.intern.model_77_142_3,60,122,2
4,amirm.mhosseini.seq1,118,94,1


In [6]:
df_designclass.head()

,name,id,designMethod,sequence,aligned-lengthafdb50,aligned-lengthcath50,aligned-lengthpdb100,binding,binding_strength,cath_detail,...,rmsdafdb50,rmsdcath50,rmsdpdb100,seqidentityafdb50,seqidentitycath50,seqidentitypdb100,ted_confidence,tm-scoreafdb50,tm-scorecath50,tm-scorepdb100
0,724928537.DBZ6 | A,mellow-cat-birch,Unknown,QVQLVESGGGLVQPGGSLRLSCAASGRTFSSYALGWFRQAPGQGLE...,108.0,100.0,112.0,False,NaN,Sandwich,...,0.96,2.41,0.82,74.7,70.7,75.6,2.0,0.92848,0.74337,0.96424
1,724928537.DBZ7 | A,rough-bat-willow,Unknown,QVQLVESGGGLVQPGGSLRLSCAASGYTFTEYPLGWFRQAPGQGLE...,99.0,93.0,103.0,False,NaN,Sandwich,...,1.21,2.45,1.11,80.6,73.2,81.4,2.0,0.91914,0.72739,0.93867
2,724928537.DBZ8 | A,wild-raven-granite,Unknown,QVQLVESGGGLVQPGGSLRLSCAASGFSLTVYGLGWFRQAPGQGLE...,116.0,102.0,113.0,False,NaN,Sandwich,...,1.31,2.06,0.82,69.4,71.7,73.9,2.0,0.95274,0.76052,0.94269
3,Ali.RASOULI.intern.model_77_142_3,shy-lion-ice,Unknown,DEGIENFVNFMKKSQEMMKGNDEERREWHAWMRVMLEKSTNKDMTE...,64.0,35.0,45.0,False,NaN,Up Down Bundle,...,2.36,3.35,3.19,23.4,20.6,19.2,2.0,0.67038,0.35974,0.45335
4,AmirM.MHosseini.seq1,mellow-tiger-lotus,Unknown,PPPPVQSHFAPCPPEHAQFCFHGTCRYVVQENKPACVCHSGWVGAR...,51.0,40.0,40.0,False,NaN,Ribbon,...,1.05,2.81,2.94,76.0,79.5,79.5,2.0,0.87019,0.39701,0.38510


In [6]:
def normalize_for_fuzzy_match(s):
    s = s.lower().strip()
    s = s.replace("=","")
    # replace separators like |, space, =, comma with underscore
    s = re.sub(r"[| :,]", "_", s)
    # remove any character that is NOT a letter, number, dot, or underscore
    s = re.sub(r"[^a-z0-9._]", "", s)
    # collapse multiple underscores
    s = re.sub(r"_+", "_", s)
    # optionally strip leading/trailing underscores
    s = s.strip("_")
    return s

def robust_normalize(s):
    s = s.lower().strip()
    # replace separators with underscore
    s = re.sub(r"[| =,:]", "_", s)
    # remove any character that is not a-z, 0-9, -, _, or .
    s = re.sub(r"[^a-z0-9._-]", "", s)
    # collapse multiple underscores
    s = re.sub(r"_+", "_", s)
    # strip leading/trailing underscores
    s = s.strip("_")
    return s

df_af2["normalized"] = df_af2["complex_id"].apply(normalize_for_fuzzy_match)
df_boltz["normalized"] = df_boltz["complex_id"].apply(normalize_for_fuzzy_match)
df_chai1["normalized"] = df_chai1["complex_id"].apply(normalize_for_fuzzy_match)
df_helixfold3["normalized"] = df_helixfold3["complex_id"].apply(normalize_for_fuzzy_match)
df_of3["normalized"] = df_of3["Complex"].apply(normalize_for_fuzzy_match)
df_protenix["normalized"] = df_protenix["complex_id"].apply(normalize_for_fuzzy_match)

df_designclass['normalized_name'] = df_designclass["name"].apply(normalize_for_fuzzy_match)


In [7]:
text1 = "bundit.b_s18.manbaritone_Seq3_EGFR-binder-70-100_beta_303_dldesign_6|Lenght=71"
text2 = "bundit.b_s18.manbaritone_seq3_egfr-binder-70-100_beta_303_dldesign_6_lenght71"

print(normalize_for_fuzzy_match(text1))
print(normalize_for_fuzzy_match(text2))

bundit.b_s18.manbaritone_seq3_egfrbinder70100_beta_303_dldesign_6_lenght71
bundit.b_s18.manbaritone_seq3_egfrbinder70100_beta_303_dldesign_6_lenght71


In [8]:
merged_af2 = df_af2.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_boltz = df_boltz.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_chai1 = df_chai1.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_helixfold3 = df_helixfold3.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_of3 = df_of3.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_protenix = df_protenix.merge(df_designclass[["normalized_name", "design_class", "binding", "binding_strength"]], left_on="normalized", right_on="normalized_name", how="left")

merged_af2 = merged_af2.drop(columns=["normalized"])
merged_boltz = merged_boltz.drop(columns=["normalized"])
merged_chai1 = merged_chai1.drop(columns=["normalized"])
merged_helixfold3 = merged_helixfold3.drop(columns=["normalized"])
merged_of3 = merged_of3.drop(columns=["normalized"])
merged_protenix = merged_protenix.drop(columns=["normalized"])

merged_af2 = merged_af2.drop(columns=["normalized_name"])
merged_boltz = merged_boltz.drop(columns=["normalized_name"])
merged_chai1 = merged_chai1.drop(columns=["normalized_name"])
merged_helixfold3 = merged_helixfold3.drop(columns=["normalized_name"])
merged_of3 = merged_of3.drop(columns=["normalized_name"])
merged_protenix = merged_protenix.drop(columns=["normalized_name"])

merged_af2.to_csv("./merged_af2.csv", index=False)
merged_boltz.to_csv("./merged_boltz.csv", index=False)
merged_chai1.to_csv("./merged_chai1.csv", index=False)
merged_helixfold3.to_csv("./merged_helixfold3.csv", index=False)
merged_of3.to_csv("./merged_of3.csv", index=False)
merged_protenix.to_csv("./merged_protenix.csv", index=False)
